In [2]:
import pandas as pd
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

import preprocessing as pp
import creation_matrices as cm
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FormatStrFormatter

In [5]:
import os
import glob
import pandas as pd

def build_all_matrices(data_dir, output_type='mirna', tumor_type='coad'):
    """
    Automatically detects all metadata sheets (normal and tumor), aggregates 
    all patient mappings, processes all physical files, and creates complete matrices.
    """
    print(f"=== Starting Complete Unified Matrix Generation for [{output_type.upper()}] ===")
    
    # 1. Automatically locate all sample sheets in the directory
    sheet_patterns = [
        os.path.join(data_dir, "**", "*sheet*.tsv"),
        os.path.join(data_dir, "*.tsv")
    ]
    
    found_sheets = []
    for pattern in sheet_patterns:
        found_sheets.extend(glob.glob(pattern, recursive=True))
    found_sheets = list(set(found_sheets)) # Remove duplicates
    
    if not found_sheets:
        print("❌ Error: No .tsv sample sheets found in the data directory.")
        return
        
    print(f"📋 Found {len(found_sheets)} metadata sheets to parse:")
    for s in found_sheets:
        print(f"   -> {os.path.basename(s)}")

    # 2. Merge all sample sheet metadata maps together
    file_metadata_map = {}
    for sheet_path in found_sheets:
        try:
            df_sheet = pd.read_csv(sheet_path, sep='\t')
            df_sheet.columns = [col.strip() for col in df_sheet.columns]
            
            # Ensure required columns exist
            if 'File Name' in df_sheet.columns and 'Sample ID' in df_sheet.columns:
                for _, row in df_sheet.iterrows():
                    filename = str(row['File Name']).strip()
                    tissue = str(row.get('Tissue Type', 'tumor')).strip().lower()
                    
                    file_metadata_map[filename] = {
                        'sample_id': str(row['Sample ID']).strip(),
                        'tissue_type': tissue
                    }
        except Exception as e:
            print(f"⚠️ Skipping parsing sheet {os.path.basename(sheet_path)} due to error: {e}")

    print(f"✅ Loaded a total of {len(file_metadata_map)} patient-file mappings from all sheets.")

    # 3. Configure file discovery based on data type (miRNA vs RNA-Seq)
    if output_type == 'mirna':
        file_pattern = "**/*.mirnas.quantification.txt"
        value_col = 'reads_per_million_miRNA_mapped'
        id_col = 'miRNA_ID'
    elif output_type == 'rna':
        file_pattern = "**/*.rna_seq.augmented_star_gene_counts.tsv"
        value_col = 'tpm_unstranded'  
        id_col = 'gene_id'
    else:
        print("❌ Error: output_type must be 'mirna' or 'rna'")
        return

    # 4. Find every data file physically present in the folders
    all_physical_files = glob.glob(os.path.join(data_dir, file_pattern), recursive=True)
    if not all_physical_files:
        # Fallback expansion check if files were modified
        fallback_ext = "*.txt" if output_type == 'mirna' else "*.tsv"
        all_physical_files = glob.glob(os.path.join(data_dir, "**", fallback_ext), recursive=True)

    print(f"📂 Detected {len(all_physical_files)} total physical expression files.")

    # 5. Extract data and sort into Normal vs Cancer collections
    normal_cohort = {}
    cancer_cohort = {}
    skipped_count = 0
    
    for file_path in all_physical_files:
        filename = os.path.basename(file_path)
        
        # Check against metadata sheets map
        if filename not in file_metadata_map:
            # Fuzzy match fallback
            fuzzy_match = [k for k in file_metadata_map.keys() if k in filename or filename in k]
            if fuzzy_match:
                meta = file_metadata_map[fuzzy_match[0]]
            else:
                skipped_count += 1
                continue
        else:
            meta = file_metadata_map[filename]
            
        sample_id = meta['sample_id']
        tissue_type = meta['tissue_type']
        
        try:
            comment_marker = '#' if output_type == 'rna' else None
            df = pd.read_csv(file_path, sep='\t', comment=comment_marker)
            
            # Remove tracking/statistical rows if GDC RNA files
            df = df[~df[id_col].astype(str).str.startswith('N_')]
            df = df.set_index(id_col)
            
            selected_metric = value_col if value_col in df.columns else df.columns[0]
            expression_series = df[selected_metric]
            
            # Route based on clinical tissue type designation
            if 'normal' in tissue_type or 'solid tissue' in tissue_type:
                target_cohort = normal_cohort
            else:
                target_cohort = cancer_cohort
                
            # Replicate protection loop
            base_id = sample_id
            rep_idx = 1
            while sample_id in target_cohort:
                rep_idx += 1
                sample_id = f"{base_id}_rep{rep_idx}"
                
            target_cohort[sample_id] = expression_series
            
        except Exception as e:
            print(f"⚠️ Error parsing file {filename}: {e}")

    print(f"⏭️ Skipped {skipped_count} files (unmapped metadata or structural sheets).")
    print(f"📊 Extracted cohorts: {len(normal_cohort)} Normal samples | {len(cancer_cohort)} Cancer samples.")

    # 6. Save matrices via Outer Join DataFrame Engine
    def compile_and_export(cohort_dict, condition_label):
        if not cohort_dict:
            print(f"ℹ️ No data collected for {condition_label.upper()} matrix compilation.")
            return
            
        final_df = pd.DataFrame(cohort_dict).fillna(0.0)
        
        if output_type == 'mirna':
            mirna_ids = final_df.index
            gene_symbols = [str(mid).replace('hsa-', '').replace('-', '') for mid in mirna_ids]
            final_df.index = pd.MultiIndex.from_arrays(
                [mirna_ids, gene_symbols], 
                names=['mirna_id', 'gene_symbol']
            )
        else:
            final_df.index.name = 'gene_id'
            
        os.makedirs('./Matrices', exist_ok=True)
        save_path = f'./Matrices/{output_type}_{condition_label}_{tumor_type}.csv'
        final_df.to_csv(save_path)
        print(f"🎉 Created: {save_path} ({final_df.shape[0]} rows x {final_df.shape[1]} columns)")

    compile_and_export(normal_cohort, 'norm')
    compile_and_export(cancer_cohort, 'canc')
    print("✅ Matrix pipeline run complete!\n")


# =====================================================================
# CONFIGURATION & RUN EXECUTION
# =====================================================================
# Update this path to where ALL your files and sheets are downloaded
GDC_DATA_DIRECTORY =  r"C:\Users\samal\Downloads\Abhinaya"
# Step 1: Generate Complete miRNA Matrices
build_all_matrices(data_dir=GDC_DATA_DIRECTORY, output_type='mirna', tumor_type='coad')

# Step 2: Generate Complete RNA-Seq Matrices (Uncomment below to run for RNA-Seq genes too!)
# build_all_matrices(data_dir=GDC_DATA_DIRECTORY, output_type='rna', tumor_type='coad')

=== Starting Complete Unified Matrix Generation for [MIRNA] ===
📋 Found 8 metadata sheets to parse:
   -> gdc_sample_sheet.2026-06-24 (4).tsv
   -> gdc_sample_sheet.2026-06-24 (1).tsv
   -> gdc_sample_sheet.2026-06-24 (6).tsv
   -> gdc_sample_sheet.2026-06-24 (8).tsv
   -> gdc_sample_sheet.2026-06-24 (2).tsv
   -> gdc_sample_sheet.2026-06-24 (5).tsv
   -> gdc_sample_sheet.2026-06-24 (7).tsv
   -> gdc_sample_sheet.2026-06-24 (3).tsv
✅ Loaded a total of 1331 patient-file mappings from all sheets.
📂 Detected 465 total physical expression files.
⏭️ Skipped 0 files (unmapped metadata or structural sheets).
📊 Extracted cohorts: 8 Normal samples | 457 Cancer samples.
🎉 Created: ./Matrices/mirna_norm_coad.csv (1881 rows x 8 columns)
🎉 Created: ./Matrices/mirna_canc_coad.csv (1881 rows x 457 columns)
✅ Matrix pipeline run complete!



In [6]:
import os
import glob
import pandas as pd

def build_perfect_matrix(data_dir, data_type='rna', tumor_type='coad'):
    """
    Builds precise, unpolluted expression matrices with distinct validation layers.
    data_type can be 'rna' or 'mirna'
    """
    print(f"=== Running Perfect Matrix Pipeline for [{data_type.upper()}] ===")
    
    # 1. Enforce strict criteria based on chosen data modality
    if data_type == 'rna':
        file_pattern = "**/*.rna_seq.augmented_star_gene_counts.tsv"
        sheet_keyword = "rna" # Target sheet identification hints if available
        id_col = 'gene_id'
        value_col = 'tpm_unstranded'
    elif data_type == 'mirna':
        file_pattern = "**/*.mirnas.quantification.txt"
        sheet_keyword = "mirna"
        id_col = 'miRNA_ID'
        value_col = 'reads_per_million_miRNA_mapped'
    else:
        print("❌ Error: data_type must be either 'rna' or 'mirna'")
        return

    # 2. Gather sample sheets
    sheet_files = list(set(glob.glob(os.path.join(data_dir, "**", "*.tsv"), recursive=True)))
    file_metadata_map = {}
    
    for sheet_path in sheet_files:
        # Ignore files that are actual data files instead of metadata sheets
        if ".rna_seq." in sheet_path or ".mirnas." in sheet_path:
            continue
        try:
            df_sheet = pd.read_csv(sheet_path, sep='\t')
            df_sheet.columns = [col.strip() for col in df_sheet.columns]
            
            if 'File Name' in df_sheet.columns and 'Sample ID' in df_sheet.columns:
                for _, row in df_sheet.iterrows():
                    fname = str(row['File Name']).strip()
                    
                    # Target filtering: skip pairing rna data to mirna configurations
                    if data_type == 'rna' and '.txt' in fname: continue
                    if data_type == 'mirna' and '.tsv' in fname: continue
                        
                    tissue = str(row.get('Tissue Type', 'tumor')).strip().lower()
                    file_metadata_map[fname] = {
                        'sample_id': str(row['Sample ID']).strip(),
                        'tissue_type': tissue
                    }
        except Exception as e:
            pass

    print(f"✅ Filtered Metadata Map compiled: {len(file_metadata_map)} valid target file rows.")

    # 3. Locate actual target files physically present
    all_physical_files = glob.glob(os.path.join(data_dir, file_pattern), recursive=True)
    if not all_physical_files:
        print(f"❌ Error: No physical files found matching pattern: {file_pattern}")
        return
    print(f"📂 Detected {len(all_physical_files)} physical data files matching [{data_type.upper()}] specification.")

    # 4. Extract data using strict unique-sample checks
    normal_cohort = {}
    cancer_cohort = {}
    
    processed_normal_samples = set()
    processed_cancer_samples = set()
    skipped_count = 0

    for file_path in all_physical_files:
        filename = os.path.basename(file_path)
        
        if filename not in file_metadata_map:
            skipped_count += 1
            continue
            
        meta = file_metadata_map[filename]
        sample_id = meta['sample_id']
        tissue_type = meta['tissue_type']
        
        is_normal = 'normal' in tissue_type or 'solid tissue' in tissue_type
        
        # Guard layer against duplicate collection
        if is_normal:
            if sample_id in processed_normal_samples: continue
        else:
            if sample_id in processed_cancer_samples: continue

        try:
            # Safely skip GDC headers if reading RNA-Seq
            comment_char = '#' if data_type == 'rna' else None
            df = pd.read_csv(file_path, sep='\t', comment=comment_char)
            
            # Remove structural/summary statistics rows
            df = df[~df[id_col].astype(str).str.startswith('N_')]
            df = df.set_index(id_col)
            
            # Extract expression vector
            metric_col = value_col if value_col in df.columns else df.columns[0]
            expression_series = df[metric_col]
            
            if is_normal:
                normal_cohort[sample_id] = expression_series
                processed_normal_samples.add(sample_id)
            else:
                cancer_cohort[sample_id] = expression_series
                processed_cancer_samples.add(sample_id)
                
        except Exception as e:
            print(f"⚠️ Error reading {filename}: {e}")

    print(f"📊 Final Cohorts: {len(normal_cohort)} Normal columns | {len(cancer_cohort)} Cancer columns.")

    # 5. Build Join Arrays and Save
    os.makedirs('./Matrices', exist_ok=True)
    
    for cohort_dict, label in [(normal_cohort, 'norm'), (cancer_cohort, 'canc')]:
        if cohort_dict:
            final_df = pd.DataFrame(cohort_dict).fillna(0.0)
            final_df.index.name = id_col
            
            # Apply uniform formatting cleanup for miRNA multi-indexes if needed
            if data_type == 'mirna':
                mirna_ids = final_df.index
                gene_symbols = [str(mid).replace('hsa-', '').replace('-', '') for mid in mirna_ids]
                final_df.index = pd.MultiIndex.from_arrays([mirna_ids, gene_symbols], names=['mirna_id', 'gene_symbol'])

            out_path = f"./Matrices/{data_type}_{label}_{tumor_type}.csv"
            final_df.to_csv(out_path)
            print(f"🎉 Created clean Matrix: {out_path} ({final_df.shape[0]} rows x {final_df.shape[1]} columns)")


# =====================================================================
# CONFIGURATION AND EXECUTION RUN
# =====================================================================
GDC_DATA_DIRECTORY =  r"C:\Users\samal\Downloads\Abhinaya"
# Step 1: Run perfect RNA-Seq processing
build_perfect_matrix(data_dir=GDC_DATA_DIRECTORY, data_type='rna', tumor_type='coad')

# Step 2: Run perfect miRNA processing
build_perfect_matrix(data_dir=GDC_DATA_DIRECTORY, data_type='mirna', tumor_type='coad')

=== Running Perfect Matrix Pipeline for [RNA] ===
✅ Filtered Metadata Map compiled: 701 valid target file rows.
📂 Detected 524 physical data files matching [RNA] specification.
📊 Final Cohorts: 41 Normal columns | 473 Cancer columns.
🎉 Created clean Matrix: ./Matrices/rna_norm_coad.csv (60660 rows x 41 columns)
🎉 Created clean Matrix: ./Matrices/rna_canc_coad.csv (60660 rows x 473 columns)
=== Running Perfect Matrix Pipeline for [MIRNA] ===
✅ Filtered Metadata Map compiled: 630 valid target file rows.
📂 Detected 465 physical data files matching [MIRNA] specification.
📊 Final Cohorts: 8 Normal columns | 453 Cancer columns.
🎉 Created clean Matrix: ./Matrices/mirna_norm_coad.csv (1881 rows x 8 columns)
🎉 Created clean Matrix: ./Matrices/mirna_canc_coad.csv (1881 rows x 453 columns)


In [8]:
import pandas as pd
import preprocessing as pp

# =====================================================================
# STEP 1: Define the import_data function explicitly inside this cell
# =====================================================================
def import_data(m_canc_path, m_norm_path, g_canc_path, g_norm_path):
    print("Reading data matrices...")
    
    # 1. Load miRNA Cancer and Normal Data
    mir_canc = pd.read_csv(m_canc_path)
    mir_norm = pd.read_csv(m_norm_path)
    
    # Safely handle single index vs multi-index formats
    if 'mirna_id' in mir_canc.columns:
        mir_canc = mir_canc.set_index(['mirna_id', 'gene_symbol'])
        mir_norm = mir_norm.set_index(['mirna_id', 'gene_symbol'])
    else:
        mir_canc = mir_canc.set_index(mir_canc.columns[0])
        mir_norm = mir_norm.set_index(mir_norm.columns[0])
        
    # 2. Load RNA-Seq Gene Data
    gene_canc = pd.read_csv(g_canc_path)
    # Set BOTH Column 0 (Ensembl) and Column 1 (Gene Symbol) as the index
    gene_canc = gene_canc.set_index([gene_canc.columns[0], gene_canc.columns[1]]) 
    
    gene_norm = pd.read_csv(g_norm_path)
    # Set BOTH Column 0 (Ensembl) and Column 1 (Gene Symbol) as the index
    gene_norm = gene_norm.set_index([gene_norm.columns[0], gene_norm.columns[1]])
    
    print("✅ All matrices loaded into system memory successfully!")
    return mir_canc, len(mir_canc), mir_norm, len(mir_norm), gene_canc, gene_norm


# =====================================================================
# STEP 2: Configure your file paths
# =====================================================================
m_canc = "./Matrices/mirna_canc_coad.csv"
m_norm = "./Matrices/mirna_norm_coad.csv"
g_canc = "./Matrices/rna_canc_coad.csv"
g_norm = "./Matrices/rna_norm_coad.csv"


# =====================================================================
# STEP 3: Execute data import (This will now run flawlessly!)
# =====================================================================
mir_canc, mir_canc_length, mir_norm, mir_norm_length, gene_canc, gene_norm = import_data(m_canc, m_norm, g_canc, g_norm)


# =====================================================================
# STEP 4: Define tumor target type and trigger preprocessing
# =====================================================================
tumor = 'coad'

preprocessed = pp.Preprocessing(
    miRNACancer=mir_canc, 
    miRNACancerlength=mir_canc,  # Pass dataframe to protect against internal .index rules
    miRNANormal=mir_norm, 
    miRNANormallength=mir_norm,  # Pass dataframe to protect against internal .index rules
    geneCancer=gene_canc, 
    geneNormal=gene_norm, 
    tumor=tumor
)

normal = preprocessed.Normal
cancer = preprocessed.Cancer

print("\n🎉 Setup Finalized: Matrices loaded, noise-filtered, and multi-omics columns aligned!")
print(f"📊 Normal Dataset Dimensions: {normal.shape}")
print(f"📊 Cancer Dataset Dimensions: {cancer.shape}")
# === Put this at the bottom of Cell 1 and RUN it ===
normal = preprocessed.Normal
cancer = preprocessed.Cancer
indices = preprocessed.Indices   # <-- Added this to match your lowercase 'indices' in Cell 2
tumor = preprocessed.tumor       # <-- Ensures 'tumor' is also defined

Reading data matrices...
✅ All matrices loaded into system memory successfully!
geneNorm (60660, 40)
geneCanc (60660, 472)
🔍 Automatically translating Ensembl IDs to Gene Names... (Takes ~10 seconds)


29 input query terms found dup hits:	[('ENSG00000175711', 2), ('ENSG00000188660', 2), ('ENSG00000215156', 2), ('ENSG00000226506', 2), ('E
1258 input query terms found no hit:	['ENSG00000002079', 'ENSG00000112096', 'ENSG00000130723', 'ENSG00000131484', 'ENSG00000132832', 'ENS


normal (24715, 41)
cancer (24715, 473)

🎉 Setup Finalized: Matrices loaded, noise-filtered, and multi-omics columns aligned!
📊 Normal Dataset Dimensions: (18771, 8)
📊 Cancer Dataset Dimensions: (18771, 452)


In [5]:
print(normal.head())
print(cancer.head())

                 TCGA-A6-2671-11A  TCGA-A6-2683-11A  TCGA-A6-2684-11A  \
ensembl                                                                 
ENSG00000000003           63.9548           87.6204           64.3955   
ENSG00000000005            0.9383            2.3156            1.6325   
ENSG00000000419           55.2409           61.5590           54.1263   
ENSG00000000457            5.3681            6.1423            5.9352   
ENSG00000000938            2.8373            2.6026            3.5623   

                 TCGA-AA-3520-11A  TCGA-AA-3525-11A  TCGA-AA-3527-11A  \
ensembl                                                                 
ENSG00000000003           53.5405           30.5298           83.1777   
ENSG00000000005            1.2938            0.6713            1.8218   
ENSG00000000419           61.7060           60.3606           62.6221   
ENSG00000000457            6.5999            6.1477            6.2446   
ENSG00000000938            4.0849            6.876

In [9]:
import os
os.makedirs('./Networks', exist_ok=True)
import creation_matrices as cm
#We create the two networks that are the sum of Euclidean and Pearson matrices
#All details of the construction are explained in creation_matrices.py script
normal_perc = [75,74,73,70,65,64,63,62,60,58,57,55,54,53,50]
cancer_perc = [65,64,63,62,60,58,57,55,54,53,50,44,40,35,30]
matrices = cm.Net(normal, cancer, indices, tumor, normal_perc, cancer_perc)
n_sum = matrices.n_sum
c_sum = matrices.c_sum

100%|████████████████████████████████████████████████████████████████████████████| 18771/18771 [12:09<00:00, 25.74it/s]


permutation calculated
Eucl coad normalized
pval: 75
normal_pears: 21772750
pval: 74
normal_pears: 21501518
pval: 73
normal_pears: 21230286
pval: 70
normal_pears: 20416592
pval: 65
normal_pears: 19060432
pval: 64
normal_pears: 18789202
pval: 63
normal_pears: 18517970
pval: 62
normal_pears: 18246738
pval: 60
normal_pears: 17704274
pval: 58
normal_pears: 17161812
pval: 57
normal_pears: 16890580
pval: 55
normal_pears: 16348116
pval: 54
normal_pears: 16076884
pval: 53
normal_pears: 15805652
pval: 50
normal_pears: 14991958
thresh pval: 50
normal_pears: 14991958
Here!Perc: 30
thresh pval: 30
cancer_pears: 4904834
Normal and cancer coad done


In [2]:
import fusion_networks as fn

In [3]:
import numpy as np

n_sum = np.load("./Networks/normal_sum_pears_eucl_coad.npy")   # your first similarity matrix
c_sum = np.load("./Networks/cancer_sum_pears_eucl_coad.npy")   # your second similarity matrix
tumor = "COAD"                 # example tumor type


In [4]:
#All details of the fusion are explained in fusion_networks.py script
fused = fn.fusion_snf(n_sum, c_sum, str(tumor))
fused_thr = fused.fused_thr

fused: 352332


In [6]:
import networkx as nx

In [8]:
#We build the graph using networkx and we save it in Gephi format
graph = nx.from_numpy_array(fused_thr)
nx.write_gexf(graph,'./Graphs/graph_'+str(tumor)+'.gexf')